In [3]:
import os
from dotenv import load_dotenv

load_dotenv()

required = ["OPENROUTER_API_KEY", "LANGSMITH_API_KEY", "LANGSMITH_TRACING", "DATABASE_URL"]
missing = [k for k in required if not os.getenv(k)]
assert not missing, f"Missing in .env: {missing}"

print("tracing:", os.getenv("LANGSMITH_TRACING"))
print("project:", os.getenv("LANGSMITH_PROJECT"))

tracing: true
project: trenz-agent


In [4]:
from langsmith import Client

print("LangSmith OK:", Client().info.version)

LangSmith OK: 0.17.20


In [5]:
import psycopg, pandas as pd

DSN = os.getenv("DATABASE_URL")

with psycopg.connect(DSN) as conn:
    df = pd.read_sql("SELECT * FROM active_listings", conn)

print(len(df), "active listings")
df[["id", "purpose", "price_pkr", "property_type", "bedrooms", "location_name"]].head()

C:\Users\Insha Khan\AppData\Local\Temp\ipykernel_24152\4057489264.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("SELECT * FROM active_listings", conn)


57 active listings


,id,purpose,price_pkr,property_type,bedrooms,location_name
0,49687025,sale,140000000,plot,NaN,DOHS Phase 1
1,49901807,sale,44500000,flat,3.0,Askari 5 - Sector F
2,47397605,sale,79000000,house,5.0,Askari 6
3,47397618,sale,80000000,house,5.0,Askari 6
4,47398147,sale,125000000,house,8.0,DOHS Phase 1


In [8]:
from langchain_openrouter import ChatOpenRouter

MODEL = "minimax/minimax-m2.7:free"

model = ChatOpenRouter(model=MODEL, temperature=0, max_retries=2)

resp = model.invoke("Reply with exactly: pipeline ok")
print(resp.content)
print("tokens:", resp.usage_metadata)

pipeline ok
tokens: {'input_tokens': 47, 'output_tokens': 59, 'total_tokens': 106, 'input_token_details': {'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'reasoning': 71}}


In [9]:
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    tools=[],
    system_prompt="You are a helpful assistant for a Karachi real estate agency.",
)

result = agent.invoke({"messages": [{"role": "user", "content": "Hello, who are you?"}]})
print(result["messages"][-1].content)

Hello! I'm a helpful assistant for a Karachi real estate agency. I'm here to help you with any questions related to real estate in Karachi, Pakistan.

Whether you're looking to:

- **Buy** a property (apartment, house, plot, commercial space)
- **Sell** your property
- **Rent** or **Lease** a property
- **Invest** in real estate
- Get information about different neighborhoods and areas in Karachi

Feel free to ask me anything, and I'll do my best to assist you!

How can I help you today?


In [10]:
result = agent.invoke({"messages": [{"role": "user", "content": "Askari 6 me kya available hai?"}]})
print(result["messages"][-1].content)

# Askari 6 - Karachi Real Estate

Assalam o Alaikum! 🙏

**Askari 6** (Askari Housing Society) Karachi mein ek masnooi residential society hai jo **Cantt Board** ke zariye manage hoti hai.

## General Information:

### 📍 Location
- Malir Area, Karachi
- Near Jinnah Avenue & Korangi Road

### 🏠 Property Types Available
- **Houses** (Various sizes: 5 Marla, 10 Marla, 1 Kanal)
- **Plots** (Residential & Commercial)
- **Commercial Properties**

### ✅ Key Features
- Gated Community
- 24/7 Security
- Parks & Recreation Areas
- Schools & Mosques
- Wide Roads

---

## ⚠️ Important Note:

Maine real-time property listing database nahi hai. **Specific availability, prices aur details** ke liye aap humari agency se directly contact karein:

📱 **Call/WhatsApp:** [Apni Agency ka Number]
🏢 **Office:** [Office Address]

Hamari team aapko:
- ✅ Current market rates
- ✅ Verified listings
- ✅ Site visits
- ✅ Legal documentation mein madad karegi

---

**Aap ko exactly kya chahiye?** 
- Plot size batao (5 

In [11]:
def fmt_pkr(n: int) -> str:
    if n >= 10_000_000:
        return f"{n/10_000_000:.2f}".rstrip("0").rstrip(".") + " Crore"
    if n >= 100_000:
        return f"{n/100_000:.2f}".rstrip("0").rstrip(".") + " Lakh"
    return f"{n:,}"

print(fmt_pkr(107_500_000), "|", fmt_pkr(135_000), "|", fmt_pkr(90_000))

10.75 Crore | 1.35 Lakh | 90,000


In [12]:
from langchain.tools import tool

@tool
def inventory_summary() -> str:
    """Summarise what property inventory Trez Enterprises currently has:
    which areas, property types, how many, and the price range in each."""
    sql = """
        SELECT loc.name AS area, l.property_type, l.purpose,
               count(*) AS n, min(l.price_pkr) AS lo, max(l.price_pkr) AS hi
        FROM listings l
        JOIN locations loc ON loc.id = l.location_id
        WHERE l.is_active
        GROUP BY 1, 2, 3
        ORDER BY n DESC
    """
    with psycopg.connect(DSN) as conn, conn.cursor() as cur:
        rows = cur.execute(sql).fetchall()

    return "\n".join(
        f"{area} — {n} {ptype}(s) for {purpose}, {fmt_pkr(lo)} to {fmt_pkr(hi)}"
        for area, ptype, purpose, n, lo, hi in rows
    )

print(inventory_summary.invoke({}))

Askari 6 — 18 house(s) for sale, 7 Crore to 9.5 Crore
Askari 5 - Sector J — 6 flat(s) for sale, 4.8 Crore to 5.5 Crore
Askari 6 — 4 flat(s) for sale, 3.9 Crore to 4.6 Crore
Emaar Panorama — 3 flat(s) for sale, 7.5 Crore to 18.95 Crore
Falcon Complex New Malir — 2 house(s) for sale, 10 Crore to 10.25 Crore
Emaar The Views — 2 flat(s) for sale, 9.6 Crore to 18.2 Crore
DOHS Phase 1 — 2 house(s) for sale, 12.5 Crore to 14.75 Crore
Gulshan-e-Iqbal — 2 flat(s) for sale, 2.49 Crore to 3.3 Crore
Navy Housing Scheme Karsaz — 2 house(s) for sale, 21 Crore to 21.5 Crore
Askari 4 — 2 flat(s) for sale, 6.9 Crore to 6.9 Crore
Askari 5 - Sector J — 2 house(s) for sale, 9.75 Crore to 9.75 Crore
Callachi Cooperative Housing Society — 1 flat(s) for sale, 2.85 Crore to 2.85 Crore
Askari 5 - Sector E — 1 flat(s) for sale, 4.2 Crore to 4.2 Crore
Memon Goth Road — 1 plot(s) for sale, 85 Lakh to 85 Lakh
HMR Waterfront — 1 flat(s) for sale, 4 Crore to 4 Crore
Cantt Bazar — 1 flat(s) for sale, 2.65 Crore to 2.

In [13]:
SYSTEM = """You are the WhatsApp assistant for Trez Enterprises, a Karachi real estate agency.

Rules:
- ONLY discuss properties returned by your tools. Never invent a listing.
- If you have no data for what they asked, say so plainly and tell them what you DO have.
- Customers write in English, Urdu or Roman Urdu. Reply in the language they used.
- Prices in Crore/Lakh, never raw digits."""

agent = create_agent(model=model, tools=[inventory_summary], system_prompt=SYSTEM)

result = agent.invoke({"messages": [{"role": "user", "content": "Askari 6 me kya available hai?"}]})
print(result["messages"][-1].content)

Askari 6 mein abhi Trez Enterprises ke paas ye options hain:

**🪴 Sale (Khareed):**

| Type | Qty | Price Range |
|------|-----|-------------|
| House | 18 | 7 Crore – 9.5 Crore |
| Flat | 4 | 3.9 Crore – 4.6 Crore |

**🏠 Rent (Kiraya):**

| Type | Qty | Price |
|------|-----|-------|
| House | 1 | 1.35 Lakh/month |
| Flat | 1 | 90,000/month |

---

Agar aapko koi specific size, location ya budget pasand hai to batao — main details check kar sakta hoon!


In [14]:
for m in result["messages"]:
    m.pretty_print()

================================ Human Message =================================

Askari 6 me kya available hai?
================================== Ai Message ==================================
Tool Calls:
  inventory_summary (call_function_pzj38oy1k80q_1)
 Call ID: call_function_pzj38oy1k80q_1
  Args:
================================= Tool Message =================================
Name: inventory_summary

Askari 6 — 18 house(s) for sale, 7 Crore to 9.5 Crore
Askari 5 - Sector J — 6 flat(s) for sale, 4.8 Crore to 5.5 Crore
Askari 6 — 4 flat(s) for sale, 3.9 Crore to 4.6 Crore
Emaar Panorama — 3 flat(s) for sale, 7.5 Crore to 18.95 Crore
Falcon Complex New Malir — 2 house(s) for sale, 10 Crore to 10.25 Crore
Emaar The Views — 2 flat(s) for sale, 9.6 Crore to 18.2 Crore
DOHS Phase 1 — 2 house(s) for sale, 12.5 Crore to 14.75 Crore
Gulshan-e-Iqbal — 2 flat(s) for sale, 2.49 Crore to 3.3 Crore
Navy Housing Scheme Karsaz — 2 house(s) for sale, 21 Crore to 21.5 Crore
Askari 4 — 2 flat(s) for

In [15]:
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model=model,
    tools=[inventory_summary],
    system_prompt=SYSTEM,
    checkpointer=InMemorySaver(),
)

cfg = {"configurable": {"thread_id": "customer-923001234567"}}

def say(text: str):
    out = agent.invoke({"messages": [{"role": "user", "content": text}]}, cfg)
    print(f"👤 {text}\n🤖 {out['messages'][-1].content}\n")

say("Assalam o alaikum, ghar dhoond raha hun")
say("Askari 6 me kya hai?")
say("aur us se sasta?")     # ← only works if it remembered Askari 6

👤 Assalam o alaikum, ghar dhoond raha hun
🤖 Aap ke liye abhi **Trez Enterprises mein jo ghar available hain** — yeh dekho:

---

🏠 **Ghar For Sale:**

| Area | Price |
|------|-------|
| **Askari 6** | 7 Crore — 9.5 Crore |
| **Falcon Complex, New Malir** | 10 Crore — 10.25 Crore |
| **DOHS Phase 1** | 12.5 Crore — 14.75 Crore |
| **Navy Housing Scheme, Karsaz** | 21 Crore — 21.5 Crore |
| **Askari 5 - Sector J** | 9.75 Crore |
| **Zamzama** | 48 Crore |

---

🏠 **Ghar For Rent:**

| Area | Rent |
|------|------|
| **Askari 6** | 1.35 Lakh/month |

---

Yeh sirf aaj ke available properties hain. Agar aap ka budget ya area different hai toh bataein — shayad kuch aur bhi ho! 

Konsa area pasand hai aap ko? 😊

👤 Askari 6 me kya hai?
🤖 Askari 6 mein abhi yeh available hai:

---

**🏠 Ghar FOR SALE:**

- **18 Ghar** — Prices: **7 Crore se 9.5 Crore** tak

**🏠 Flat FOR SALE:**

- **4 Flat** — Prices: **3.9 Crore se 4.6 Crore** tak

---

**🏠 Flat FOR RENT:**

- **1 Flat** — Rent: **90,000/mont

In [16]:
SYSTEM = """You are the WhatsApp assistant for Trez Enterprises, a Karachi real estate agency.

## Formatting — WhatsApp, NOT markdown
- NEVER use tables, # headers, or --- lines. They do not render on WhatsApp.
- Bold is *single asterisks*, never **double**.
- Short lines. Blank line between items. Max ~8 lines per reply.
- Prices as Crore / Lakh, never raw digits.

## Truthfulness
- ONLY mention properties your tools returned. Never invent one.
- If asked for a house and only flats/plots match, SAY SO explicitly:
  "Us se sasta koi ghar nahi hai, lekin flats hain."
  Never silently switch property type.
- If there is nothing, say so and name the areas you DO cover.

## Conversation
- Reply in the language they used (English / Urdu / Roman Urdu).
- Don't dump the whole inventory. Ask ONE qualifying question first:
  budget, area, or buy-vs-rent — whichever is missing.
- Always know whether they want to BUY or RENT before quoting prices."""

agent = create_agent(
    model=model,
    tools=[inventory_summary],
    system_prompt=SYSTEM,
    checkpointer=InMemorySaver(),
)

cfg = {"configurable": {"thread_id": "test-2"}}
say("Assalam o alaikum, ghar dhoond raha hun")

👤 Assalam o alaikum, ghar dhoond raha hun
🤖 Walaikum assalam! 🏠

Kya aap ghar khareedna chahte hain ya rent par?

Aur kab tak budget hai? Tells me your range and preferred area, I'll check what's available.



In [17]:
import re

def as_whatsapp(text: str) -> str:
    """Strip what WhatsApp can't render, so you see the real thing."""
    text = re.sub(r"^\s*\|.*\|\s*$", "[TABLE — BROKEN ON WHATSAPP]", text, flags=re.M)
    text = re.sub(r"^\s*[-–—]{3,}\s*$", "", text, flags=re.M)
    text = re.sub(r"^#{1,6}\s*", "", text, flags=re.M)
    text = text.replace("**", "*")
    return re.sub(r"\n{3,}", "\n\n", text).strip()

print(as_whatsapp(result["messages"][-1].content))

Askari 6 mein abhi Trez Enterprises ke paas ye options hain:

*🪴 Sale (Khareed):*
[TABLE — BROKEN ON WHATSAPP]
[TABLE — BROKEN ON WHATSAPP]
[TABLE — BROKEN ON WHATSAPP]
[TABLE — BROKEN ON WHATSAPP]
*🏠 Rent (Kiraya):*
[TABLE — BROKEN ON WHATSAPP]
[TABLE — BROKEN ON WHATSAPP]
[TABLE — BROKEN ON WHATSAPP]
[TABLE — BROKEN ON WHATSAPP]

Agar aapko koi specific size, location ya budget pasand hai to batao — main details check kar sakta hoon!


In [18]:
import re, unicodedata

GENERIC = {"town","city","phase","block","sector","road","society",
           "complex","housing","scheme","area","new","old"}

def norm(s: str) -> str:
    """Same normalisation the aliases were stored with."""
    s = unicodedata.normalize("NFKD", s or "").encode("ascii", "ignore").decode()
    return re.sub(r"\s+", " ", re.sub(r"[^a-z0-9]+", " ", s.lower())).strip()

def _windows(q: str, maxn: int = 4) -> list[str]:
    """'ask 6 me ghar chahiye' -> every 1..4-word slice.
    Needed because similarity('ask 6', <whole sentence>) is low — we must
    compare the alias against the PIECE of the message that names a place."""
    w = q.split()
    return list({" ".join(w[i:i+n]) for n in range(1, maxn+1)
                 for i in range(len(w)-n+1)} | {q})

_SQL = """
WITH q(win) AS (SELECT unnest(%(windows)s::text[]))
SELECT a.location_id, l.name, l.depth, a.alias_norm, a.source,
       similarity(a.alias_norm, q.win) AS score,
       (SELECT count(*) FROM listings li
         WHERE li.is_active
           AND li.location_id IN (SELECT id FROM descendants_of(l.id))) AS stock
FROM location_aliases a
JOIN locations l ON l.id = a.location_id
CROSS JOIN q
WHERE similarity(a.alias_norm, q.win) >= %(thr)s
"""

def resolve_location(text: str, threshold: float = 0.45) -> list[dict]:
    q = norm(text)
    if not q:
        return []
    qt = {t for t in q.split() if len(t) >= 4}

    with psycopg.connect(DSN) as conn, conn.cursor() as cur:
        rows = cur.execute(_SQL, {"windows": _windows(q), "thr": threshold}).fetchall()

    best: dict[int, dict] = {}
    for loc_id, name, depth, alias, source, score, stock in rows:
        at = {t for t in alias.split() if len(t) >= 4}
        # GUARD: must share a MEANINGFUL word. Without this,
        # "2 bed in bahria town" matches "Gadap Town" on the word "town".
        if at and not any(t not in GENERIC for t in (at & qt)) and score < 0.80:
            continue
        cand = {"location_id": loc_id, "name": name, "depth": depth,
                "via": alias, "source": source,
                "score": float(score), "spec": len(alias.split()), "stock": stock}
        cur_best = best.get(loc_id)
        if cur_best is None or (cand["score"], cand["spec"]) > (cur_best["score"], cur_best["spec"]):
            best[loc_id] = cand

    # ties break toward the MORE SPECIFIC place, then the one with stock
    return sorted(best.values(),
                  key=lambda c: (-c["score"], -c["spec"], -c["depth"], -c["stock"]))[:3]

def decide(cands: list[dict]) -> str:
    if not cands:
        return "NO_MATCH"
    a, b = cands[0], (cands[1] if len(cands) > 1 else None)
    if a["score"] >= 0.60 and (b is None or a["score"] - b["score"] > 0.12):
        return "ACCEPT"
    if a["score"] >= 0.60 and b and a["score"] == b["score"] and a["spec"] > b["spec"]:
        return "ACCEPT"
    if a["score"] >= 0.60 and b and a["stock"] > 0 and b["stock"] == 0:
        return "ACCEPT"
    return "ASK"

In [19]:
for text in ["flat in askari 5", "ask 6 me ghar chahiye", "3 bed apartment sector j",
             "malir cantt", "jauhar me flat chahiye", "koi flat hai defence me",
             "karsaz me ghar", "askri 5 me house", "house near garden east",
             "2 bed in bahria town"]:
    c = resolve_location(text)
    top = f'{c[0]["name"]} ({c[0]["score"]:.2f} via "{c[0]["via"]}", stock={c[0]["stock"]})' if c else "—"
    print(f'{decide(c):<9} {text:<28} {top}')

ACCEPT    flat in askari 5             Askari 5 (1.00 via "askari 5", stock=10)
ACCEPT    ask 6 me ghar chahiye        Askari 6 (1.00 via "ask 6", stock=24)
ACCEPT    3 bed apartment sector j     Askari 5 - Sector J (1.00 via "sector j", stock=8)
ACCEPT    malir cantt                  Malir Cantonment (1.00 via "malir cantt", stock=38)
ACCEPT    jauhar me flat chahiye       Gulistan-e-Jauhar (1.00 via "jauhar", stock=2)
ACCEPT    koi flat hai defence me      DHA Defence (1.00 via "defence", stock=6)
ACCEPT    karsaz me ghar               Navy Housing Scheme Karsaz (1.00 via "karsaz", stock=2)
ASK       askri 5 me house             Askari 5 (0.56 via "ask 5", stock=10)
NO_MATCH  house near garden east       —
NO_MATCH  2 bed in bahria town         —


In [20]:
SORTS = {"price_asc": "l.price_pkr ASC", "price_desc": "l.price_pkr DESC",
         "area_desc": "l.area_sqyd DESC NULLS LAST", "newest": "l.first_seen_on DESC"}

def search_listings(purpose, property_type=None, location_ids=None,
                    min_price=None, max_price=None, min_beds=None, max_beds=None,
                    features=None, sort="price_asc", limit=5) -> list[dict]:
    if purpose not in ("sale", "rent"):
        raise ValueError("purpose must be 'sale' or 'rent' — never guess it")

    where, p = ["l.is_active", "l.purpose = %(purpose)s"], {"purpose": purpose, "limit": limit}

    if property_type:
        where.append("l.property_type = ANY(%(ptypes)s)")
        p["ptypes"] = [property_type] if isinstance(property_type, str) else list(property_type)

    if location_ids:
        # one parent id expands to all its descendants
        where.append("""l.location_id IN (
            SELECT d.id FROM unnest(%(locs)s::int[]) AS r(rid),
                             LATERAL descendants_of(r.rid) AS d)""")
        p["locs"] = list(location_ids)

    for col, key, op, val in [("price_pkr", "min_price", ">=", min_price),
                              ("price_pkr", "max_price", "<=", max_price),
                              ("bedrooms",  "min_beds",  ">=", min_beds),
                              ("bedrooms",  "max_beds",  "<=", max_beds)]:
        if val is not None:
            where.append(f"l.{col} {op} %({key})s")
            p[key] = val

    if features:  # must have ALL requested features
        where.append("""(SELECT count(*) FROM listing_features f
                          WHERE f.listing_id = l.id AND f.feature = ANY(%(feats)s))
                        = cardinality(%(feats)s)""")
        p["feats"] = list(features)

    sql = f"""
      SELECT l.id, l.price_pkr, l.property_type, l.bedrooms, l.bathrooms,
             l.area_sqyd, loc.name AS area, l.title,
             COALESCE((SELECT array_agg(f.feature ORDER BY f.feature)
                       FROM listing_features f WHERE f.listing_id = l.id), '{{}}') AS features,
             (SELECT count(*) FROM listing_media m WHERE m.listing_id = l.id) AS photos
      FROM listings l
      LEFT JOIN locations loc ON loc.id = l.location_id
      WHERE {' AND '.join(where)}
      ORDER BY {SORTS.get(sort, SORTS['price_asc'])}
      LIMIT %(limit)s"""

    with psycopg.connect(DSN) as conn, conn.cursor() as cur:
        cur.execute(sql, p)
        cols = [d.name for d in cur.description]
        return [dict(zip(cols, r)) for r in cur.fetchall()]

In [21]:
print("A) askari 6, flat, 4 bed, <= 4 Cr")
for r in search_listings("sale", "flat", [21109], max_price=40_000_000, min_beds=4):
    print("  ", r["id"], fmt_pkr(r["price_pkr"]), r["bedrooms"], "bed", r["area"])

print("\nB) cheapest HOUSE for sale  <- the 518x trap")
for r in search_listings("sale", "house", limit=3):
    print("  ", r["id"], fmt_pkr(r["price_pkr"]), r["area"])

print("\nC) everything for RENT")
for r in search_listings("rent", limit=5):
    print("  ", r["id"], fmt_pkr(r["price_pkr"]), r["property_type"], r["area"])

print("\nD) Askari 5 parent expands to its sectors")
rs = search_listings("sale", "flat", [6655], limit=10)
print("  ", len(rs), "results across", sorted({r["area"] for r in rs}))

print("\nE) purpose guard")
try:
    search_listings("buy")
except ValueError as e:
    print("   raised:", e)

A) askari 6, flat, 4 bed, <= 4 Cr
   52755299 3.9 Crore 4 bed Askari 6
   52369071 3.9 Crore 4 bed Askari 6

B) cheapest HOUSE for sale  <- the 518x trap
   50815130 7 Crore Askari 6
   54579219 7.4 Crore Askari 6
   53934444 7.49 Crore Askari 6

C) everything for RENT
   53934205 90,000 flat Askari 6
   53934091 1.35 Lakh house Askari 6

D) Askari 5 parent expands to its sectors
   8 results across ['Askari 5 - Sector E', 'Askari 5 - Sector F', 'Askari 5 - Sector J']

E) purpose guard
   raised: purpose must be 'sale' or 'rent' — never guess it


In [22]:
@tool
def find_location(text: str) -> str:
    """Resolve an area name a customer typed (e.g. 'ask 6', 'malir cantt', 'jauhar')
    into a location_id. ALWAYS call this before searching by area.
    Returns candidates with a location_id, or says no match."""
    c = resolve_location(text)
    if not c:
        return f"No location matching '{text}'. We only cover parts of Karachi."
    verdict = decide(c)
    lines = [f'{x["name"]} (location_id={x["location_id"]}, {x["stock"]} listings)' for x in c]
    if verdict == "ACCEPT":
        return f"Resolved to {lines[0]}"
    return "Ambiguous — ask the customer which they mean:\n" + "\n".join(lines)


@tool
def find_properties(purpose: str, property_type: str | None = None,
                    location_id: int | None = None, max_price: int | None = None,
                    min_price: int | None = None, min_beds: int | None = None,
                    features: list[str] | None = None, limit: int = 5) -> str:
    """Search Trez Enterprises inventory.

    purpose: REQUIRED, 'sale' or 'rent'. Never guess — ask the customer if unclear.
    property_type: 'house', 'flat' or 'plot'.
    location_id: from find_location. Never invent this number.
    min_price/max_price: PKR. 1 crore = 10000000, 1 lakh = 100000.
    features: any of west_open, east_open, corner, park_facing, sea_view,
              furnished, on_installments, brand_new, brigadier_house, ground_floor.
    """
    rows = search_listings(purpose, property_type, [location_id] if location_id else None,
                           min_price, max_price, min_beds, features=features, limit=limit)
    if not rows:
        return "No matching properties."
    return "\n".join(
        f'id={r["id"]} | {fmt_pkr(r["price_pkr"])} | {r["property_type"]} | '
        f'{r["bedrooms"] or "-"} bed | {r["area_sqyd"] or "-"} sq.yd | {r["area"]} | '
        f'{", ".join(r["features"][:4])} | {r["photos"]} photos'
        for r in rows)

In [23]:
agent = create_agent(
    model=model,
    tools=[inventory_summary, find_location, find_properties],
    system_prompt=SYSTEM + """

## Tools
- Area mentioned? call find_location FIRST. Never invent a location_id.
- find_properties needs purpose ('sale' or 'rent'). If the customer hasn't said
  which, ASK — do not assume. Rent prices are per month.
- Quote only ids the tools returned.""",
    checkpointer=InMemorySaver(),
)

cfg = {"configurable": {"thread_id": "demo-1"}}
say("ask 6 me 4 bed flat chahiye, budget 4 crore tak")

👤 ask 6 me 4 bed flat chahiye, budget 4 crore tak
🤖 Yeh 2 matching flats mili hain Askari 6 mein, 4 crore tak:

1. *4 Bed Flat — 356 sq.yd*
   Price: 3.9 Crore
   Features: Brand New, Prayer Room, Servant Quarter, Study Room

2. *4 Bed Flat — 328 sq.yd*
   Price: 3.9 Crore
   Features: Brand New, Park Facing, Prayer Room, Servant Quarter

Kon si zyada pasand aayi? Detail chahiye ya visit arrange karun?



In [ ]:
from agents.graph import build_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = build_agent(InMemorySaver())

: 